#### Simple Gen AI APP Using Langchain

In [3]:
import os
import tqdm as notebook_tqdm
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [4]:
## Data Ingestion - Langchain websote doc link to scrape data from
from langchain_community.document_loaders import WebBaseLoader

In [8]:
loader = WebBaseLoader("https://www.langchain.com/blog/tuning-deep-agents-different-models")
loader

In [10]:
docs=loader.load()
print(f"Number of documents: {len(docs)}")
docs

Number of documents: 1


[Document(metadata={'source': 'https://www.langchain.com/blog/tuning-deep-agents-different-models', 'title': 'Tuning Deep Agents to Work Well with Different Models', 'description': 'Deep Agents was previously designed in a generic way to work well across model families. Today we’re adding model-specific profiles to adjust prompts, tools, and middleware. We ship profiles for OpenAI, Anthropic, and Google models out of the box, which we see leads to a 10–20 point jump on a subset of tau2-bench over the default harness.', 'language': 'en'}, page_content='Tuning Deep Agents to Work Well with Different Models\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceDeploymentShip and scale agents in productionFleetAgents for the whole companyOpen Source FrameworksdeepagentsBuild long-running agents for complex taskslangchainQuick

In [11]:
### Load Data => Docs => Divide into chunks documents => text => vectors => Vector Embedding => VectorDB => Retrieval => LLM response
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

In [12]:
documents

[Document(metadata={'source': 'https://www.langchain.com/blog/tuning-deep-agents-different-models', 'title': 'Tuning Deep Agents to Work Well with Different Models', 'description': 'Deep Agents was previously designed in a generic way to work well across model families. Today we’re adding model-specific profiles to adjust prompts, tools, and middleware. We ship profiles for OpenAI, Anthropic, and Google models out of the box, which we see leads to a 10–20 point jump on a subset of tau2-bench over the default harness.', 'language': 'en'}, page_content='Tuning Deep Agents to Work Well with Different Models\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceDeploymentShip and scale agents in productionFleetAgents for the whole companyOpen Source FrameworksdeepagentsBuild long-running agents for complex taskslangchainQuick

In [13]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="gpt-4o", dimensions=1024)

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)
vectorstoredb

In [ ]:
# Query from vectorstoredb
query = "What are the different models mentioned in the article?"
result=vectorstoredb.similarity_search(query)
result

In [15]:
from langchain_openai import ChatOpenAI 
llm=ChatOpenAI(model="gpt-4o")

In [ ]:
## Retrieval chain, Document Chain, LLM Chain
from langchain.chains.combine_documents import create_stuff_document_chain
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant. Use the following retrieved documents to answer the question.
    {context}
    Question: {question}
    Answer:
    """
)
doc_chain = create_stuff_document_chain(llm=llm, prompt=prompt)
doc_chain

In [ ]:
doc_chain.run(input_documents=result, question=query)

In [ ]:
from langchain_core.documents import Document
doc_chain.invoke({
    "input": "What are the different models mentioned in the article?",
    
})